# Notebook 04a: Static Class-Mean-Features (CMF) Unlearning (`04a_cmf_static.ipynb`)

### Objectives & Workflow:
1. **CMF Head Reconstruction & Tying**: Reconstruct classifier weights directly from centered & normalized class-mean features at each epoch.
2. **Representation-Level Unlearning**: Freeze the CMF head and optimize only the encoder $\theta$ using the unlearning objective across models (`resnet18` / `vit11m`) and seeds `[0, 1, ..., 9]`.
3. **Validation-Driven Multi-Stage Training Engine**: Continued training over staged learning rates with patience, evaluating on validation and selecting checkpoints based on feature erasure (`1 - LP_f`, `1 - NCC_f`) under retain accuracy constraints.

In [ ]:
# ============================================================
# SETUP: Clone repo from GitHub & add to Python path
# ============================================================
import sys
import os
import subprocess

REPO_URL = "https://github.com/tiensinh2/Unlearning_experiments_1.git"
REPO_DIR = "Unlearning_experiments_1"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL}...")
    try:
        from git import Repo
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gitpython", "-q"])
        from git import Repo
    Repo.clone_from(REPO_URL, REPO_DIR)
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository already exists at {REPO_DIR}")
    from git import Repo
    repo = Repo(REPO_DIR)
    origin = repo.remotes.origin
    origin.pull()
    print("Pulled latest changes")

repo_path = os.path.abspath(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
    print(f"Added {repo_path} to sys.path")
else:
    print(f"{repo_path} already in sys.path")

In [ ]:
import os
import copy
import json
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import datasets

from experiment_utils import (
    build_classifier,
    get_dataset_transforms,
    IndexedDataset,
    extract_features,
    compute_output_accuracy,
    evaluate_three_tier_metrics,
    visualize_feature_space_and_boundaries,
    MODEL_CONFIGS
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

## 1. CMF Head Reconstruction Utility

In [ ]:
@torch.no_grad()
def reconstruct_cmf_head(model, dataloader, num_classes=10, device="cuda"):
    """
    Computes class-mean features, centers them against the global mean, normalizes, and assigns to linear weights.
    """
    feats, targets = extract_features(model.encoder, dataloader, device=device)
    class_means = []
    
    for c in range(num_classes):
        mask = (targets == c)
        if mask.sum() > 0:
            mean = feats[mask].mean(axis=0)
        else:
            mean = np.zeros(feats.shape[1])
        class_means.append(mean)
        
    class_means = np.stack(class_means, axis=0) # [C, D]
    global_mean = class_means.mean(axis=0, keepdims=True)
    centered = class_means - global_mean
    norms = np.linalg.norm(centered, axis=1, keepdims=True) + 1e-8
    normalized_weights = centered / norms # [C, D]
    
    # Assign to model linear layer
    weight_tensor = torch.tensor(normalized_weights, dtype=torch.float32, device=device)
    model.head.linear.weight.data.copy_(weight_tensor)
    if model.head.linear.bias is not None:
        model.head.linear.bias.data.zero_()
    
    # Freeze classifier head
    model.head.linear.weight.requires_grad = False
    if model.head.linear.bias is not None:
        model.head.linear.bias.requires_grad = False

## 2. Validation-Driven Continued Training Engine for 04a (from `model_instruction.md`)

In [ ]:
DATASET_NAME = "cifar10"
SPLIT_SEED = 123  # Single fixed seed for dataset partitioning
MODEL_NAME = "resnet18"  # Options: resnet18, vit11m

cfg = MODEL_CONFIGS[DATASET_NAME]
MODEL_SEEDS = cfg["model_seeds"]  # [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
BATCH_SIZE = cfg["batch_size"]    # 256
NUM_CLASSES = 10 if DATASET_NAME in ["cifar10", "mnist", "fashion_mnist"] else (100 if DATASET_NAME == "cifar100" else 5)

# Staged learning schedule
STAGES = [
    {"lr": 2e-4, "max_epochs": 4, "patience": 2},
    {"lr": 5e-5, "max_epochs": 3, "patience": 2},
]
DELTA_RETAIN = 0.05  # Retain accuracy constraint δ
MIN_DELTA = 0.002

train_tf, eval_tf = get_dataset_transforms(DATASET_NAME)
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)

os.makedirs("./artifacts/checkpoints", exist_ok=True)
os.makedirs("./artifacts/metrics", exist_ok=True)
os.makedirs("./artifacts/splits", exist_ok=True)

# Search path in Kaggle input or local artifacts
kaggle_split = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
local_split = f"./artifacts/splits/{DATASET_NAME}_seed_{SPLIT_SEED}_split.json"
split_file = kaggle_split if os.path.exists(kaggle_split) else local_split
if not os.path.exists(split_file):
    raise FileNotFoundError(f"Split artifact not found at {kaggle_split} or {local_split}. Run Notebook 01 first.")
print(f"Using split artifact: {split_file}")
    
with open(split_file, "r") as f:
    split_info = json.load(f)

# Datasets & Loaders
class RandomTargetDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, num_classes):
        self.base_dataset = base_dataset
        self.num_classes = num_classes
    def __len__(self):
        return len(self.base_dataset)
    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img = item[0]
        rand_label = np.random.randint(0, self.num_classes)
        if len(item) == 3:
            return img, rand_label, item[2]
        return img, rand_label

forget_dataset = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), train_tf)
forget_dataset_rl = RandomTargetDataset(forget_dataset, NUM_CLASSES)

retain_dataset = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), train_tf)
combined_dataset = ConcatDataset([forget_dataset_rl, retain_dataset])
train_loader = DataLoader(combined_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

retain_eval_set = IndexedDataset(Subset(raw_train, split_info["train_r_indices"]), eval_tf)
forget_eval_set = IndexedDataset(Subset(raw_train, split_info["train_f_indices"]), eval_tf)
val_set = IndexedDataset(Subset(raw_train, split_info["val_indices"]), eval_tf)
test_set = IndexedDataset(Subset(raw_test, split_info["test_indices"]), eval_tf)

retain_eval_loader = DataLoader(retain_eval_set, batch_size=BATCH_SIZE, shuffle=False)
forget_eval_loader = DataLoader(forget_eval_set, batch_size=BATCH_SIZE, shuffle=False)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

for seed in MODEL_SEEDS:
    kaggle_ckpt = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.pt"
    local_ckpt = f"./artifacts/checkpoints/original_{MODEL_NAME}_seed_{seed}.pt"
    orig_ckpt = kaggle_ckpt if os.path.exists(kaggle_ckpt) else local_ckpt

    kaggle_metrics = f"/kaggle/input/datasets/kiethe/cifar10-experimentnb1/original_{MODEL_NAME}_seed_{seed}.json"
    local_metrics = f"./artifacts/metrics/original_{MODEL_NAME}_seed_{seed}.json"
    orig_metrics_file = kaggle_metrics if os.path.exists(kaggle_metrics) else local_metrics
    
    if not os.path.exists(orig_ckpt) or not os.path.exists(orig_metrics_file):
        raise FileNotFoundError(f"Original artifacts for model {MODEL_NAME} seed {seed} missing (checked Kaggle and local paths). Run Notebook 01 first.")
    print(f"Using original ckpt: {orig_ckpt} | metrics: {orig_metrics_file}")
        
    with open(orig_metrics_file, "r") as f:
        orig_val_retain = json.load(f)["validation_metrics"]["output_retain"]
        
    print(f"\n==================================================")
    print(f"--- Running 04a: CMF Static | Model: {MODEL_NAME} | Seed: {seed} ---")
    print(f"==================================================")
    
    # Load original model
    model = build_classifier(model_name=MODEL_NAME, num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(orig_ckpt, map_location=device))
    
    criterion = nn.CrossEntropyLoss()
    best_snapshot = None
    start_time = time.time()
    
    # --- Multi-Stage Continued Training Loop ---
    for stage_idx, stage in enumerate(STAGES):
        if best_snapshot is not None:
            model.load_state_dict(best_snapshot["state"])
            
        # Only encoder parameters are trainable; head is reconstructed & frozen
        optimizer = optim.SGD(model.encoder.parameters(), lr=stage["lr"], momentum=0.9, weight_decay=5e-4)
        no_gain = 0
        
        for epoch in range(1, stage["max_epochs"] + 1):
            # Reconstruct CMF head at beginning of epoch
            reconstruct_cmf_head(model, train_loader, num_classes=NUM_CLASSES, device=device)
            
            # Train Encoder
            model.encoder.train()
            model.head.eval()
            for images, targets in train_loader:
                images, targets = images.to(device), targets.to(device)
                optimizer.zero_grad()
                out = model(images)
                loss = criterion(out, targets)
                loss.backward()
                optimizer.step()
                
            # Validation evaluation for selection and stage control
            val_metrics = evaluate_three_tier_metrics(
                model=model, train_loader=train_loader, retain_eval_loader=retain_eval_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
            )
            
            # Score = Feature-level erasure (1 - LP_f, 1 - NCC_f) + Retain accuracy
            score = 0.5 * (1.0 - val_metrics["lp_forget"]) + 0.5 * (1.0 - val_metrics["ncc_forget"]) + val_metrics["output_retain"]
            eligible = (val_metrics["output_retain"] >= orig_val_retain - DELTA_RETAIN)
            
            print(f"Stage {stage_idx+1} Ep {epoch} | Val Retain: {val_metrics['output_retain']:.4f} | LP Forget: {val_metrics['lp_forget']:.4f} | NCC Forget: {val_metrics['ncc_forget']:.4f} | Score: {score:.4f} (Eligible: {eligible})")
            
            if eligible and (best_snapshot is None or score > best_snapshot["score"] + MIN_DELTA):
                best_snapshot = {"score": score, "metrics": val_metrics, "state": copy.deepcopy(model.state_dict())}
                no_gain = 0
            else:
                no_gain += 1
                
            if no_gain >= stage["patience"]:
                print(f"Early stopping triggered in stage {stage_idx+1}")
                break
                
    # Restore final best checkpoint
    if best_snapshot is not None:
        model.load_state_dict(best_snapshot["state"])
        
    runtime = time.time() - start_time
    save_ckpt = f"./artifacts/checkpoints/cmf_04a_{MODEL_NAME}_seed_{seed}.pt"
    torch.save(model.state_dict(), save_ckpt)
    
    # Evaluate once on test set
    test_metrics = evaluate_three_tier_metrics(
        model=model, train_loader=train_loader, retain_eval_loader=test_loader, forget_eval_loader=forget_eval_loader, num_classes=NUM_CLASSES, seed=seed, device=device
    )
    
    res = {
        "variant": "04a_cmf_static",
        "model_name": MODEL_NAME,
        "seed": seed,
        "split_seed": SPLIT_SEED,
        "runtime_sec": runtime,
        "validation_metrics": best_snapshot["metrics"] if best_snapshot else val_metrics,
        "test_metrics": test_metrics
    }
    res_path = f"./artifacts/metrics/cmf_04a_{MODEL_NAME}_seed_{seed}.json"
    with open(res_path, "w") as f:
        json.dump(res, f, indent=2)
        
    print(f"04a Test Retain Output Acc: {test_metrics['output_retain']:.4f} | Forget: {test_metrics['output_forget']:.4f}")
    print(f"04a Test LP Forget Acc:     {test_metrics['lp_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_lp']:.4f})")
    print(f"04a Test NCC Forget Acc:    {test_metrics['ncc_forget']:.4f} (Illusion Gap: {test_metrics['illusion_gap_ncc']:.4f})")
    print(f"Saved 04a checkpoint & metrics to {res_path}")
    
    # Feature space & decision boundary visualization for first seed
    if seed == MODEL_SEEDS[0]:
        vis_save = f"./artifacts/plots/features_04a_cmf_{MODEL_NAME}_seed_{seed}.png"
        visualize_feature_space_and_boundaries(
            model=model,
            retain_loader=test_loader,
            forget_loader=forget_eval_loader,
            num_classes=NUM_CLASSES,
            forget_class=0,
            title=f"Feature Space & Linear Output: 04a CMF Static ({MODEL_NAME})",
            device=device,
            save_path=vis_save
        )